## 操作系统概述：定义·四大特征·发展史

> 2026-10-11 · 来源：学习对话（块1 从头重过）

- **我的理解**：用户要求「从头开始操作系统」，从概述与运行机制起步。
- **知识点**：
  - **OS 定义**：管理硬件资源、控制程序执行、提供人机接口与程序接口的**系统软件**。四大功能 = 处理机管理、存储管理、文件管理、设备管理。
  - **四大特征**：并发、共享、虚拟、异步。**并发与共享互为存在条件**——无并发就没有资源争用，无共享并发就没有意义。
  - **发展史（被什么逼出来）**：CPU 与 I/O 速度差 → 单道批处理 CPU 空等 I/O → **多道程序设计**（内存同时驻留多道程序交替执行，提高利用率）→ 分时（时间片轮转 + 交互）→ 实时（及时 + 可靠）。
- **项目映射**：Linux 多进程并发访问同一文件就是「并发+共享」的日常直觉来源。边界：408 只考概念辨析，不考 Linux 细节。
- **考试侧重点**：四特征辨析、多道/分时/实时比较（选择，☆常规）。
- **易错点**：
  - 「虚拟」= 物理实体 → 逻辑上多份（一个 CPU 虚拟成多个、一块盘虚拟成多个逻辑卷），不是「虚拟出来不存在」。
  - 「异步」= 并发+共享下程序「走走停停」、执行不可再现，这是特征不是缺陷。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-12


## 双态切换与中断/异常

> 2026-10-11 · 来源：学习对话（块1 从头重过）

- **我的理解**：用户要求「从头开始操作系统」，双态是后面一切章节的入口。
- **知识点**：
  - CPU 分**用户态（目态）**与**核心态（管态）**；**特权指令**只在核心态可执行：I/O 启动、置时钟、修改 PSW、修改页表基址寄存器、清内存等。
  - **用户态→核心态的诱因只有三类**：系统调用（陷入）、外中断、内异常。**核心态→用户态**只发生在中断返回时（内核改程序状态字）。
  - **中断（外中断）vs 异常（内中断）**判断标准：**是否由当前指令引发**。
    - 外中断：与当前指令无关——时钟中断、I/O 完成中断。
    - 内中断（异常）：当前指令引发——**陷入 trap**（系统调用入口）、**故障 fault**（如缺页）、**终止 abort**（如非法指令）。
- **项目映射**：x86 ring0/ring3、Linux 的中断描述符表。边界：408 只考概念与流程判断，不考具体汇编与寄存器。
- **考试侧重点**：特权指令判断、切换时机判断（★核心，必考选择）。
- **易错点**：
  - **陷入指令不是特权指令**——它恰恰在用户态执行（否则用户根本调不了系统调用），但它引发切换。
  - **进程切换不是用户态→核心态的诱因**——进程切换发生在核心态内部，不是切换的原因。
  - 「读时钟」「取数」是读类操作，不是特权指令；「写时钟」「启动 I/O」才是。
- **例题/错题**：待补充（验收走自主刷题判卷）
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-12


## 系统调用与 OS 体系结构

> 2026-10-11 · 来源：学习对话（块1 从头重过）

- **我的理解**：用户要求「从头开始操作系统」，系统调用是用户态进入内核的唯一主动入口。
- **知识点**：
  - **系统调用过程（6 步）**：用户程序传参 → 执行**陷入指令**（用户态执行，但引发切换）→ 切核心态 → 查系统调用入口表 → 执行内核服务例程 → 返回用户态、取返回值。
  - **体系结构**：大内核（宏内核）/ 微内核 / 分层式 / 模块化 / 外核（○了解）。微内核 = 内核只留最基本功能（中断、调度、IPC、基本内存管理），其余移到用户态服务。
- **项目映射**：Linux `syscall` 指令即陷入入口；jos 的系统调用走 `int` 门。边界：408 只考流程与判断，不考寄存器传参细节。
- **考试侧重点**：流程排序题、库函数 vs 系统调用判断（★核心）。
- **易错点**：
  - **库函数 ≠ 系统调用**：`strlen` 纯用户态不陷入；`printf` 本身是库函数，内部触发的 `write` 才是系统调用。
  - 传参发生在**执行陷入指令之前**（靠寄存器/栈传参，陷入后无法再访问用户栈拿参数是常见干扰项）。
- **例题/错题**：待补充（验收走自主刷题判卷）
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-12


## x86 双态与中断/异常：ring 切换机制（jos 对照）

> 2026-10-12 · 来源：学习对话（块1 深化：特权指令专题）

- **我的理解**：用户 jos 做过但没吃透，想借特权指令专题把 x86 的 trap/interrupt 分类与 ring 切换一次搞完。
- **知识点**：
  - x86 分 ring0–3，Linux/jos 实际只用 0（内核）与 3（用户）。CPL 存于 CS 低 2 位；**改 CPL 必须经远跳/门/iret，用户程序无法直接加载 CS**——这就是 408「用户态→核心态只能经中断/异常」的硬件强制版本。
  - **3→0**：只能走受控入口——`int n`（系统调用门 DPL=3 才允许用户调用）、异常、NMI。**0→3**：`iret`（恢复 CS:EIP 时 CPL 随 CS 变回 3）。
  - **x86 中断/异常分类**：
    - 中断（异步，与当前指令无关）：可屏蔽（INTR，受 EFLAGS.IF 控制）/ 不可屏蔽 NMI。
    - 异常（同步，当前指令引发）：**fault**（指令执行前检测，返回地址指向本指令——除0 #DE、缺页 #PF）、**trap**（执行后触发——int3、into、int 0x80）、**abort**（机器检查等严重错误）。
  - **术语对齐（重要）**：408 教材的「陷入 trap」常特指系统调用那条内中断；x86 的 trap 只是异常三分法之一，宽窄不同，别拿 jos 记忆硬套 408。
  - **特权指令的真实对应**：408 说「修改 PSW 是特权」→ x86 里 `cli`/`sti`（改 EFLAGS.IF 位）、`lidt`、`lgdt`、`mov cr0` 都只能在 ring0 执行。
- **项目映射**（jos，路径经 learning-408/README.md 定位并实读核实）：
  - `jos/inc/trap.h`：向量号全景——0–19 为处理器定义（T_DIVIDE=0、T_GPFLT=13、T_PGFLT=14、T_NMI=2），**T_SYSCALL=48 是 JOS 自选**，IRQ_OFFSET=32（PIC 重映射，时钟中断 = int 32）。
  - `jos/kern/trapentry.S`：`TRAPHANDLER`（CPU 自动推 error code）vs `TRAPHANDLER_NOEC`（软件 `pushl $0` 补齐栈帧）——**有没有 error code 是 fault 类的物理痕迹**。
  - `jos/lib/syscall.c:23`：`asm volatile("int %1\n")` —— 用户态主动陷入，即 408 的「陷入指令」本体。
  - `jos/kern/env.c:495 env_pop_tf`：`iret` 弹出 Trapframe 完成 0→3 的切换（`popal` → `iret`）。
- **考试侧重点**：408 只考「哪些指令特权 / 何时切换 / 中断异常分类」（★核心题型 1/2/4）；DPL/CPL/RPL 门检查细节**不考**（Intel 手册 Vol.3 §6.13，想深读看 Vol.3A Ch.6）。
- **易错点**：除0 是 **fault** 不是 trap（返回地址指向本除法指令，可重执行）；int3 是 **trap**（执行后触发）——同为「断点」类概念，触发时机相反。
- **待解决问题**：jos trap 全链路走读（trapentry.S → _alltraps → trap() → trap_dispatch），用户此前未吃透的部分，是否展开待用户定。
- **复习计划**：R1 @ 2026-10-13


## trapframe 分层与 error code：不是入参，是断点

> 2026-10-12 · 来源：学习对话（块1 深化 Q1）

- **我的理解**：用户记得 error code「只是推了一个 int」，疑惑为什么不能把入参放进构造的 trapframe；并提出 trapframe 两个要点——执行断点 + 用户/内核 trap 流程不同。
- **知识点**：
  - **error code 不是入参通道**。它确实是「一个 int」，但内容是硬件按异常类型**固定格式**填的诊断位：
    - 段类异常（#GP 13、#SS 12 等）= 段选择子索引 + EXT/IDT/TI 标志位；
    - **缺页 #PF 14 = 3 个位**：bit0 P（0=页不在内存，1=页在但权限不够）、bit1 W（0=读、1=写）、bit2 U（0=内核态访问、1=用户态访问）。
    - 用途：让 handler 不用再探测就知道「为什么错」——COW 写时复制靠 W 位决定复制还是共享，U 位决定往用户还是内核的错误路径走。
  - **trapframe 是四层拼起来的**（jos `inc/trap.h` Trapframe 注释原文分了层）：
    1. 软件层：`tf_regs`（pushal 压全部通用寄存器）+ es/ds（`_alltraps` 补压）；
    2. 软件层：`tf_trapno`——**向量号硬件不推**，是 `TRAPHANDLER` 宏 `pushl $(num)` 压的；
    3. 硬件层（注释 "defined by x86 hardware"）：`tf_err`、eip、cs、eflags——CPU 按死的格式压；
    4. 硬件层（注释 "only when crossing rings"）：ss、esp——**只在发生特权级切换时才压**。
  - **入参其实确实在 trapframe 里，但走的是第 1 层**：系统调用参数放寄存器，pushal 把寄存器整包存进 `tf_regs`，内核从 `tf->tf_regs` 里读——这是「顺便被捕获」，不是 error code 槽位的职能。error code 不能当参数用：它只在部分异常存在、位置夹在硬件格式中间、每类异常含义不同，没有稳定调用约定。
- **项目映射**（路径实读核实）：
  - `jos/inc/trap.h` Trapframe/UTrapframe 结构注释：分层真源；UTrapframe 的 `utf_fault_va`（缺页虚拟地址）+ `utf_err` 供用户态 handler 用。
  - `jos/kern/env.c:495 env_pop_tf`：`addl $0x8,%%esp` 跳过的正是 tf_trapno + tf_errcode 两个软件层字段。
  - `jos/kern/trap.c:323`：`if ((tf->tf_cs & 3) == 3)` —— **用户/内核 trap 的分流就是查保存下来的 CS 低 2 位（CPL）**；:330 用户 trap 进来先 `lock_kernel()`。
  - `jos/kern/trap.c:183`：`ts->ts_esp0 = &percpu_kstacks[...]` —— 用户 trap 换栈的内核栈顶来自 TSS。
- **考试侧重点**：408 不考 trapframe 内存布局；考的是「中断响应过程」的顺序感（关中断→保存断点→引出服务例程）。
- **易错点**：以为向量号也是硬件压的（错，x86 上是软件宏压的）；以为 error code 可以传参（错，诊断位不是约定参数）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## 内核态 vs 特权指令：一个硬币的两面（调度为什么必须 ring0）

> 2026-10-12 · 来源：学习对话（块1 深化 Q2）

- **我的理解**：用户疑惑「内核态和特权指令两个概念割裂」——操作页表要特权能理解，但进程调度为什么也要内核态？时钟中断或 yield 引发的调度真的需要特权指令吗？
- **知识点**：
  - **两个概念不割裂，是同一件事的两面**：内核态（CPL=0）的定义就是「特权指令合法 + 内核内存可访问」的执行状态。特权状态（CR3、EFLAGS.IF、I/O 端口、页表内存、内核栈）是**需要保护的资产**；特权指令是**碰这些资产的唯一工具**；ring0 是**持证资格**。不存在「在 ring3 里做调度」的选项，因为调度改写的每一个寄存器都是特权寄存器。
  - **进程切换 = 一串特权指令**（jos `env_run`，实读 `kern/env.c:535`）：`lcr3(PADDR(env_pgdir))`（mov cr3，换地址空间 + 冲 TLB）→ `unlock_kernel()`（动 EFLAGS.IF，PSW 类）→ `env_pop_tf` 的 **iret**（写 CS 本身就是最核心的特权操作，0→3 全靠它）。
  - **时钟中断 handler 无条件是内核工作**：至少要 `outb` 应答 PIC（I/O 指令=特权）；jos 空闲调度循环就是 `sti` + `hlt`（`kern/sched.c:85-95`）——**hlt 在 ring3 执行直接 #GP**。
  - **「内核是自己的进程」是误解**：jos 没有内核进程。内核代码运行在**触发者的上下文**里，借的是该 CPU 的内核栈（每 CPU 一个，`trap.c:183` 的 TSS esp0）。内核代码和用户代码取指执行没有本质区别——区别只有 CPL 决定「哪些指令合法、哪些内存可访问」。
  - **用户直觉「内核内部也区分危险代码」方向对，但硬件只给一档**：ring0 内部所有特权指令全部合法，内部分级是**软件纪律**（jos 的大内核锁、Linux 的自旋锁/preempt 计数）。再往下的分级靠虚拟化：ring -1 / VT-x root mode（○了解，408 不考）。
- **项目映射**：上列均已实读核实（learning-408/jos）。
- **考试侧重点**：408 考「哪些操作需要核心态」（访页表、I/O、置时钟、开关中断、进程切换语境判断）——判据：**碰特权状态即需核心态**，与「这个操作看起来危不危险」无关。
- **易错点**：把「调度是个管理逻辑」当成可以用户态做的事——调度器代码本身不危险，**它操纵的页表/env 表/内核栈全是特权资产**；把内核当成一个特殊进程（错：内核不是进程，是运行在被中断进程上下文里的一套代码）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## 缺页时间线走一遍：error code 到底是什么、不是什么

> 2026-10-12 · 来源：学习对话（块1 深化 Q3：error code 没看懂，补讲）

- **我的理解**：用户重申「error code 只是推了一个 int」没看懂它和入参的关系，要求接着讲透。
- **知识点（时间线）**：用户程序写一个地址 → MMU 查页表发现问题 → CPU 抛 #PF（向量 14）→ 压栈序列：因 ring3→ring0，先从 TSS 取内核栈（ss0:esp0），再压 ss、esp（跨 ring 才有）→ eflags、cs、eip → **#PF 属于「有 error code 的异常」，再压一个 32 位 err** → 软件接力：TRAPHANDLER 压 trapno、_alltraps 压段寄存器与 pushal。
  - **err 的内容（#PF）= 3 个诊断位**：P（0=页不在，1=页在但权限不够）、W（0=读、1=写）、U（0=内核态、1=用户态）。例：用户写一个在内存里的 COW 共享页 → err=0b111=7；用户读一个缺页 → 0b100=4。
  - **为什么不能当入参**：① 只在部分异常存在（#DE 除0 就没有，软件 push $0 补齐格式）；② 含义随异常类型变，没有跨调用的稳定约定；③ **你最想要的那个「参数」——缺页地址——根本不在 err 里，在 CR2 寄存器**，内核用 `rcr2()` 读（jos `kern/trap.c:368`，`inc/x86.h:153` 定义）。err 是「为什么错」，CR2 才是「哪里错」。
  - **入参的真身在 tf_regs**：系统调用参数放寄存器，pushal 把寄存器整包捕获进 trapframe 第 1 层，内核从 `tf->tf_regs` 读——顺便捕获，不是 err 槽位的职能。
  - **err 位怎么用（预埋内存章）**：COW 判「要不要真正复制」只看 W 位与页是否共享；U 位决定错误路径走用户还是内核。
- **项目映射**（实读核实）：`kern/trap.c:363-377 page_fault_handler`：`fault_va = rcr2()` → `(tf->tf_cs & 3) == 0` 则内核态缺页直接 panic，用户态缺页走 upcall（用户异常栈上构造 UTrapframe，带 utf_fault_va 与 utf_err）。
- **考试侧重点**：408 不考 err 位与 CR2；考「中断响应过程」顺序与「中断隐指令」概念（关中断→保存断点→引出服务程序）。
- **易错点**：把 err 当参数通道；以为向量号是硬件压的（x86 上是 TRAPHANDLER 宏压的）；以为缺页地址在 err 里（在 CR2）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## 内核内部分级的真轴：并行性，而不是特权级（用户洞察）

> 2026-10-12 · 来源：学习对话（块1 深化 Q4）

- **我的理解**（保留原话）：「内核态的代码能不能放到 ring3 本身没有意义，内核代码本来就该对自己的硬件全权负责，所以真正的问题是有些操作可以不 lock_kernel 让多个用户态进程同时 int 进入来提升性能，内核代码的区分应该是能不能并行而不是需不需要特权级。」——**这个洞察是对的，而且是内核工程史的主线**。
- **知识点**：
  - **两根轴正交**：特权级轴解决「用户 vs 内核」的**安全边界**（防外人）；锁/并行轴解决「CPU vs CPU、中断 vs 进程」的**内部一致性**（防自己人打架）。ring0 之内不再有特权分级——分级换了一根轴：**按「碰不碰共享状态」划分**。
  - **jos 的现状（教学简化）**：每 CPU 一个内核栈（`trap.c:183` percpu_kstacks）已经为并行铺了路——多个 CPU 可以同时各陷进自己的内核栈；但 `trap.c:330` 用户 trap 一律 `lock_kernel()`（**大内核锁**），把严肃内核工作重新串行化。
  - **用户说的「有些操作可以不 lock_kernel」正是 Linux 演进史**：BKL（大内核锁，2.6 早期）→ 细粒度锁 → per-CPU 数据 → RCU/无锁。判据恰好就是「这段代码碰什么共享状态」：纯读、per-CPU、无共享 → 可并行；碰全局表 → 加锁。
  - **补一个有趣事实强化正交性**：原子指令 `xchg`/`cmpxchg` **不是特权指令**——ring3 也能用（用户态锁库、futex 全靠它）。所以「并行的机器」也不需要特权；两根轴彻底独立。
  - **408 对应**：这根轴在 408 里就是「同步与互斥」章（临界资源、互斥、PV、死锁）——概念层考试，不考内核锁实现。
- **项目映射**：上列 trap.c 行号均实读核实。
- **考试侧重点**：408 考并行/互斥的**概念与 PV 计算**（★核心，进程模块块4）；「哪些操作需核心态」走特权级轴（概述章），两根轴的题不要混。
- **易错点**：把「需要在内核态」与「需要加锁」当成一回事——前者防用户，后者防并发；以为原子指令是特权（不是）。
- **待解决问题**：待补充（进程模块块4 同步互斥学到时回来对号）
- **复习计划**：R1 @ 2026-10-13


## error code 的生成与消费时机（补问）

> 2026-10-12 · 来源：学习对话（块1 深化 Q5）

- **我的理解**：用户问「error code 就是 Intel 规定的提前帮我分类的 fault 类型对吧？#PF 的话，这个 error code 是啥时候去判断的？」
- **知识点**：
  - **表述修正**：精神对（硬件预先诊断，OS 不用再探测），但它不是「类别编号」（不是 1=缺页 2=越权 的枚举），而是**按位打包的诊断位**，每位含义由 Intel **按异常类型分别规定**——#PF 是 P/W/U，#GP 是段选择子+标志位；同一个槽不同异常语义不同。且不是所有 fault 都有（#DE 就没有）。
  - **P 位的真正价值——把缺页劈成两种命运**：P=0 页不在 → 正常慢路径，OS 分配/换入（**demand paging、懒分配全靠它**：malloc 大片不真分配，等 P=0 缺页再给）；P=1 页在但越权 → 程序 bug 或**故意的陷阱**（COW：写共享页触发，OS 查 W 位决定复制）。没有 P 位，OS 每次缺页都得重查页表区分这两种情况。
  - **两段时机**：
    - **生成（分类）**：MMU 走页表失败的那一瞬——**硬件做**，失败原因置位后随 eip/cs 一起压栈。OS 还没开始跑，分类已完成。
    - **消费（判断）**：OS 在 handler 里按位读。jos 时间线：`trap()` 先看 tf_cs&3 分流（用 cs 不用 err）→ `page_fault_handler`（trap.c:363）`rcr2()` 拿地址 → 内核态缺页 panic / 用户态缺页把 `utf_fault_va + utf_err` 打包成 UTrapframe 递给用户 handler（`env_pgfault_upcall`）→ **位级别的分诊发生在具体策略里**：demand paging 看 P，COW 看 W。
- **项目映射**：`kern/trap.c:363-377`、`inc/trap.h` UTrapframe 均实读核实。**注意**：本仓 jos 只到 lab3，COW 的位级消费者（user/fork.c 的 pgfault 查 err&FEC_WR）属于 lab4，**该部分代码尚无**。
- **比喻锚**：err = 急诊分诊台的标签（内科/外伤），CR2 = 病历主诉（哪里疼），handler 策略 = 诊室医生（按标签+主诉决定治法）。分类在分诊台（硬件），判断在诊室（软件）。
- **易错点**：把 err 当枚举类别号；以为 OS 参与了 err 的生成（纯硬件）；以为缺页处理策略只看 err（地址要另读 CR2）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## [复习] OS 概述：定义·四特征·发展史 —— R1 复盘

> 2026-10-12 · 来源：R1 到期复习（用户选择直读易错点，对话不出题；变式重做由自主刷题承担）

- **易错点复盘**：
  - 并发 ≠ 并行：并发=宏观同时微观交替（单核即可）；并行=物理同时（需多核）。OS 语境「同时」多指并发。
  - 「并发与共享互为存在条件」不是废话：无共享→各玩各的并发无意义；无并发→共享无争用。
  - 虚拟 = 物理一份 → 逻辑多份（CPU/磁盘/打印机皆可），不是「不存在」。
  - 异步（走走停停、不可再现）是并发+共享的**代价与特征**，不是缺陷。
  - 多道程序目的 = 提高资源利用率（掩盖 I/O 延迟）；**交互性是分时**（时间片+交互），及时+可靠是实时。三者目的别混。
- **结果**：复盘读式通过 → 升档
- **复习计划**：R2 @ 2026-10-15


## [复习] 双态切换与中断/异常 —— R1 复盘

> 2026-10-12 · 来源：R1 到期复习（用户选择直读易错点，对话不出题；变式重做由自主刷题承担）

- **易错点复盘**：
  - **陷入指令不是特权指令**——它恰在用户态执行（否则用户调不了），但引发切换。
  - 特权指令判据：写类/全局（置时钟、启 I/O、改 PSW、清内存、改页表基址）= 特权；**读类（读时钟、取数）不是**。
  - 用户态→核心态诱因**只有三类**：系统调用（陷入）、外中断、内异常；**进程切换不是诱因**——它发生在核心态内部。
  - 库函数 ≠ 系统调用：`strlen` 纯用户态；`printf` 内部的 write 才是。
  - 中断 vs 异常一条线：**是否当前指令引发**。异常三兄弟：trap 执行后触发（int3）、fault 执行前可重执行（缺页、除0）、abort 严重不可返回。
  - 核心态→用户态只发生在**中断返回**（内核改 PSW）；反向是硬件强制单入口。
- **结果**：复盘读式通过 → 升档
- **复习计划**：R2 @ 2026-10-15


## [复习] 系统调用与 OS 体系结构 —— R1 复盘

> 2026-10-12 · 来源：R1 到期复习（用户选择直读易错点，对话不出题；变式重做由自主刷题承担）

- **易错点复盘**：
  - 流程 6 步**顺序**：传参 → 陷入 → 切核心态 → 查入口表 → 执行服务例程 → 返回取值。**传参在陷入之前**（靠寄存器/用户栈，陷入后内核读保存的寄存器现场）。
  - 大内核 = 性能好（一切在核心态，无 IPC 开销）；微内核 = 可靠性/扩展性好（服务在用户态，代价是频繁 IPC）。
  - 判断题高频：printf/scanf **会**触发系统调用，strlen/strcpy **不会**。
- **结果**：复盘读式通过 → 升档
- **复习计划**：R2 @ 2026-10-15


## [CS149] 并行计算的效率观：FAST 不等于 EFFICIENT

> 2026-10-22 · 来源：CS149 项目复习（真源 modules/408/CS149.md）

- **FAST != EFFICIENT**：在 10 个核上拿到 2 倍加速只说明程序变快了，硬件用得好不好是另一回事。整门课都在考这个区分。
- **三个性能敌人**：① 负载不均（静态分块遇上代价不均的工作）② SIMD divergence（一个 gang 的耗时 = 最慢 lane 的耗时）③ 算术强度太低（如 saxpy 每 3 个元素只做 2 次运算 → 带宽受限，加核救不了）。
- **判断瓶颈是算力还是带宽**是并行优化的第一基本功；工具形态随硬件升级：CPU 计时器 → GPU 的 NCU 报告 → 加速器的 MFU。
- **存储层次的控制权**：CPU cache 由硬件管 → GPU shared memory 由程序员声明 → Trainium 的 SBUF/PSUM 要显式 DMA。越往上，程序员越要为数据布局负责。
- **正确性的两条不变式**：原子性（读-改-写不可分）+ 顺序（对同一位置的更新必须按输入序）。关键洞察：**顺序只约束同一个位置**，不重叠的部分完全自由 —— 并行度就藏在这句话里。
- **优化四方向**：增加并行度 / 减少通信 / 隐藏延迟 / 缓解竞争。任何一次「改快了」都要能归到其中之一。


## 冯·诺依曼 vs 哈佛：Hack 是一个活的反例

> 2026-10-22 · 来源：nand2tetris 项目复习（modules/408/NAND2TETRIS.md）
- 冯·诺依曼的核心：**指令与数据同等地位、共存于同一存储器** → 程序可以被当作数据加载与修改（这是「操作系统引导程序」的物理前提）
- nand2tetris 的 Hack 是**哈佛结构**：ROM32K（指令）与 RAM（数据）是**两条独立地址空间**，CPU 用 `pc` 取指、用 `A` 取数
- 哈佛的收益：取指与取数**并行**，单周期可行（天然无取指/访存的结构冒险）
- 哈佛的代价：**不能自修改代码，程序必须预先烧入 ROM** —— 没有「把程序加载进内存再执行」这回事
- 408 判断题套路：「指令存储器与数据存储器分开」→ 哈佛；「指令与数据共用同一存储器」→ 冯·诺依曼；「Von Neumann 机上取指与访存会争用总线」→ 对
- 记忆锚点：**冯氏换来的「可加载性」，恰好是 Hack 缺的东西** —— 两者的取舍是同一枚硬币的两面


## [CS149] 并行性能的三个定量工具：Amdahl、work-span、roofline

> 2026-10-22 · 来源：CS149 项目复习（真源 modules/408/CS149.md）

- **Amdahl 定律**：speedup = 1 / (s + (1-s)/p)。串行比例钉死加速比上限 —— s=0.25 时核数再多上限也是 4 倍；想提到 100 倍，s 必须压到 1% 以下。（408 计算题口径）
- **work-span 模型**：T_p <= W/p + D。D 是关键路径长 —— 依赖链的长度，加核逃不出它。Amdahl 是「串行部分独占 D」的特例。
- **roofline**：可达到性能 = min(峰值算力, 带宽 x 算术强度)。屋脊点 = 峰值算力/带宽（H100 FP16 约 300 FLOP/Byte）。**AI 落在墙左看带宽（fusion/tiling/量化），落在墙右看算力（占用率/Tensor Core）—— 方向搞反，优化全白干。**
- **LLM decode 为什么 memory-bound 的公式化答案**：decode 每 token 读全部权重、几乎不算，算术强度约 1 FLOP/Byte，深埋带宽墙下。
